# Step 4a — LoRA Finetuning
DINOv2 · LoRA rank ablation (r=2,4,8,16) · vs block-unfreezing · SPair-71k

In [ ]:
# Cell 0 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_ROOT = '/content/drive/MyDrive/semantic_correspondence'
RESULTS_DIR = f'{DRIVE_ROOT}/results/step4/lora'
WEIGHTS_DIR = f'{DRIVE_ROOT}/weights/finetuned/lora'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)
print('Drive mounted.')

In [ ]:
# Cell 1 — Install + clone
!pip install -q timm pandas
import os, subprocess, sys
REPO_PATH = '/content/semantic_correspondence'
if not os.path.exists(REPO_PATH):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/YOUR_USER/semantic_correspondence.git',
                    REPO_PATH], check=False)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)
print('Ready.')

In [ ]:
# Cell 2 — Imports + config
import torch, numpy as np, json, time, os, math, copy
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

SPAIR_BASE    = f'{DRIVE_ROOT}/datasets/SPair-71k/SPair-71k'
PAIR_ANN_PATH = f'{SPAIR_BASE}/PairAnnotation'
LAYOUT_PATH   = f'{SPAIR_BASE}/Layout'
IMAGE_PATH    = f'{SPAIR_BASE}/JPEGImages'
DATASET_SIZE  = 'large'
PCK_ALPHA     = 0.1
THRESHOLDS    = [0.05, 0.1, 0.2]
DINOV2_W      = f'{DRIVE_ROOT}/weights/dinov2_vitb14_pretrain.pth'
IMG_SIZE      = 518
PATCH_SIZE    = 14
TEMPERATURE   = 10.0
LR            = 1e-4
NUM_EPOCHS    = 5
PATIENCE      = 2

In [ ]:
# Cell 3 — Inline utility functions
class Normalize:
    def __init__(self, image_keys):
        self.image_keys = image_keys
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    def __call__(self, sample):
        for key in self.image_keys:
            sample[key] /= 255.0
            sample[key] = self.normalize(sample[key])
        return sample

def read_img(path):
    img = np.array(Image.open(path).convert('RGB'))
    return torch.tensor(img.transpose(2, 0, 1).astype(np.float32))

class SPairDataset(Dataset):
    def __init__(self, pair_ann_path, layout_path, image_path, dataset_size, pck_alpha, datatype):
        self.ann_files = open(os.path.join(layout_path, dataset_size, datatype + '.txt')).read().split('\n')
        self.ann_files = self.ann_files[:len(self.ann_files) - 1]
        self.pair_ann_path = pair_ann_path
        self.datatype = datatype
        self.image_path = image_path
        self.transform = Normalize(['src_img', 'trg_img'])
    def __len__(self): return len(self.ann_files)
    def __getitem__(self, idx):
        ann_file = self.ann_files[idx] + '.json'
        with open(os.path.join(self.pair_ann_path, self.datatype, ann_file)) as f:
            ann = json.load(f)
        category = ann['category']
        src_img = read_img(os.path.join(self.image_path, category, ann['src_imname']))
        trg_img = read_img(os.path.join(self.image_path, category, ann['trg_imname']))
        sample = {'src_imname': ann['src_imname'], 'trg_imname': ann['trg_imname'],
                  'src_imsize': src_img.size(), 'trg_imsize': trg_img.size(),
                  'trg_bbox': ann['trg_bndbox'], 'category': category,
                  'src_img': src_img, 'trg_img': trg_img,
                  'src_kps': torch.tensor(ann['src_kps']).float(),
                  'trg_kps': torch.tensor(ann['trg_kps']).float(),
                  'kps_ids': ann['kps_ids']}
        return self.transform(sample)

def extract_dense_features(model, img_tensor, training=False):
    ctx = torch.no_grad() if not training else torch.enable_grad()
    with ctx:
        fd = model.forward_features(img_tensor)
        pt = fd['x_norm_patchtokens']
        B, N, D = pt.shape
        H = W = int(N ** 0.5)
        return pt.reshape(B, H, W, D)

def pixel_to_patch_coord(x, y, original_size, patch_size=14, resized_size=518):
    sx = resized_size / original_size[0]
    sy = resized_size / original_size[1]
    px = int(x * sx // patch_size)
    py = int(y * sy // patch_size)
    mx = resized_size // patch_size - 1
    return min(max(px, 0), mx), min(max(py, 0), mx)

def patch_to_pixel_coord(patch_x, patch_y, original_size, patch_size=14, resized_size=518):
    xr = patch_x * patch_size + patch_size / 2
    yr = patch_y * patch_size + patch_size / 2
    return xr * original_size[0] / resized_size, yr * original_size[1] / resized_size

def find_best_match_argmax(s, width):
    idx = s.argmax().item()
    return idx % width, idx // width

def compute_pck_spair71k(pred_points, gt_points, bbox, threshold):
    pred = np.array(pred_points)
    gt   = np.array(gt_points)
    dist = np.sqrt(np.sum((pred - gt) ** 2, axis=1))
    norm = max(bbox[2] - bbox[0], bbox[3] - bbox[1])
    nd   = dist / norm
    mask = nd <= threshold
    return float(np.mean(mask) * 100), mask, nd

def compute_cross_entropy_loss(src_features, tgt_features, src_kps, tgt_kps,
                                src_imsize, tgt_imsize, patch_size, resized_size,
                                temperature=10.0):
    _, H, W, D = tgt_features.shape
    tgt_flat = tgt_features.reshape(H * W, D)
    total_loss = torch.tensor(0.0, device=src_features.device)
    count = 0
    src_sz = (src_imsize[2], src_imsize[1])
    tgt_sz = (tgt_imsize[2], tgt_imsize[1])
    for i in range(src_kps.shape[0]):
        px, py = pixel_to_patch_coord(src_kps[i, 0].item(), src_kps[i, 1].item(),
                                       src_sz, patch_size, resized_size)
        tpx, tpy = pixel_to_patch_coord(tgt_kps[i, 0].item(), tgt_kps[i, 1].item(),
                                          tgt_sz, patch_size, resized_size)
        gt_idx = tpy * W + tpx
        query = src_features[0, py, px, :]
        sim = F.cosine_similarity(query.unsqueeze(0), tgt_flat, dim=1)
        logits = sim * temperature
        total_loss = total_loss + F.cross_entropy(logits.unsqueeze(0),
                                                   torch.tensor([gt_idx], device=logits.device))
        count += 1
    return total_loss / max(count, 1)

# --- LoRA utilities ---
class LoRALinear(nn.Module):
    def __init__(self, original_linear, r=4, alpha=1.0):
        super().__init__()
        d_in  = original_linear.in_features
        d_out = original_linear.out_features
        self.original = original_linear
        self.original.weight.requires_grad = False
        if self.original.bias is not None:
            self.original.bias.requires_grad = False
        self.lora_A = nn.Parameter(torch.empty(r, d_in))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.lora_B = nn.Parameter(torch.zeros(d_out, r))
        self.scale = alpha / r
    def forward(self, x):
        return self.original(x) + (x @ self.lora_A.T @ self.lora_B.T) * self.scale

def inject_lora(model, r=4, alpha=4.0, target_modules=('qkv', 'proj')):
    for param in model.parameters():
        param.requires_grad = False
    to_replace = [
        (name, module)
        for name, module in model.named_modules()
        if isinstance(module, nn.Linear) and any(t in name for t in target_modules)
    ]
    for name, module in to_replace:
        parts = name.rsplit('.', 1)
        parent = model
        if len(parts) > 1:
            for part in parts[0].split('.'):
                parent = getattr(parent, part)
        setattr(parent, parts[-1], LoRALinear(module, r=r, alpha=alpha))
    print(f'Injected LoRA (r={r}, alpha={alpha}) into {len(to_replace)} layers')
    return model

def count_trainable_params(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    pct = round(100.0 * trainable / total, 4) if total > 0 else 0.0
    return {'trainable': trainable, 'total': total, 'percentage': pct}

def unfreeze_last_n_blocks(model, n_blocks):
    for p in model.parameters():
        p.requires_grad = False
    if hasattr(model, 'blocks'):
        for block in model.blocks[-n_blocks:]:
            for p in block.parameters():
                p.requires_grad = True
    if hasattr(model, 'norm'):
        for p in model.norm.parameters():
            p.requires_grad = True

def train_epoch_lora(model, dataloader, optimizer, device, epoch,
                      img_size=518, patch_size=14, temperature=10.0, scheduler=None):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for batch in dataloader:
        src_t = F.interpolate(batch['src_img'].to(device),
                               size=(img_size, img_size), mode='bilinear', align_corners=False)
        tgt_t = F.interpolate(batch['trg_img'].to(device),
                               size=(img_size, img_size), mode='bilinear', align_corners=False)
        src_f = extract_dense_features(model, src_t, training=True)
        tgt_f = extract_dense_features(model, tgt_t, training=True)
        loss = compute_cross_entropy_loss(
            src_f, tgt_f, batch['src_kps'][0].to(device), batch['trg_kps'][0].to(device),
            batch['src_imsize'][0], batch['trg_imsize'][0],
            patch_size=patch_size, resized_size=img_size, temperature=temperature
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        total_loss += loss.item()
        n_batches += 1
    avg_loss = total_loss / max(n_batches, 1)
    print(f'  Epoch {epoch}: avg_loss={avg_loss:.4f}')
    return avg_loss

def validate(model, val_dataset, device, img_size=518, patch_size=14,
             threshold=0.1, max_samples=500):
    model.eval()
    pck_scores = []
    with torch.no_grad():
        for idx, sample in enumerate(val_dataset):
            if idx >= max_samples:
                break
            src_t = F.interpolate(sample['src_img'].unsqueeze(0).to(device),
                                   size=(img_size, img_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(sample['trg_img'].unsqueeze(0).to(device),
                                   size=(img_size, img_size), mode='bilinear', align_corners=False)
            src_sz = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            sf = extract_dense_features(model, src_t)
            tf = extract_dense_features(model, tgt_t)
            _, H, W, D = tf.shape
            tf_flat = tf.reshape(H * W, D)
            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            trg_bbox = sample['trg_bbox']
            preds = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_sz, patch_size, img_size)
                sim = F.cosine_similarity(sf[0, py, px, :].unsqueeze(0), tf_flat, dim=1)
                mx, my = find_best_match_argmax(sim, W)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_sz, patch_size, img_size)
                preds.append([rx, ry])
            pck, _, _ = compute_pck_spair71k(preds, trg_kps.tolist(), trg_bbox, threshold)
            pck_scores.append(pck)
    model.train()
    return float(np.mean(pck_scores)) if pck_scores else 0.0

def train_with_scheduler(model, train_loader, val_dataset, device,
                          img_size=518, patch_size=14, temperature=10.0,
                          lr=1e-4, num_epochs=5, warmup_steps=100,
                          patience=2, results_dir='.'):
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=1e-4
    )
    total_steps = num_epochs * len(train_loader)
    sched1 = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=1.0/max(warmup_steps,1), end_factor=1.0, total_iters=warmup_steps
    )
    sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(total_steps - warmup_steps, 1)
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer, schedulers=[sched1, sched2], milestones=[warmup_steps]
    )
    best_pck = -1.0
    no_improve = 0
    history = []
    for epoch in range(1, num_epochs + 1):
        loss = train_epoch_lora(model, train_loader, optimizer, device, epoch,
                                 img_size, patch_size, temperature, scheduler)
        val_pck = validate(model, val_dataset, device, img_size, patch_size)
        history.append({'epoch': epoch, 'loss': loss, 'val_pck': val_pck})
        print(f'  Val PCK@0.10: {val_pck:.2f}%')
        if val_pck > best_pck:
            best_pck = val_pck
            no_improve = 0
            ckpt_path = os.path.join(results_dir, 'best_checkpoint.pth')
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'optimizer_state_dict': optimizer.state_dict(),
                        'val_pck': val_pck}, ckpt_path)
            print(f'  Saved best (PCK={best_pck:.2f}%)')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop at epoch {epoch}.')
                break
    return best_pck, history

def evaluate_model(model, dataset, device, thresholds, patch_size, resized_size):
    per_img = []
    model.eval()
    with torch.no_grad():
        for idx, sample in enumerate(dataset):
            src_t = F.interpolate(sample['src_img'].unsqueeze(0).to(device),
                                   size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            tgt_t = F.interpolate(sample['trg_img'].unsqueeze(0).to(device),
                                   size=(resized_size, resized_size), mode='bilinear', align_corners=False)
            src_sz = (sample['src_imsize'][2], sample['src_imsize'][1])
            tgt_sz = (sample['trg_imsize'][2], sample['trg_imsize'][1])
            sf = extract_dense_features(model, src_t)
            tf = extract_dense_features(model, tgt_t)
            _, H, W, D = tf.shape
            tf_flat = tf.reshape(H * W, D)
            src_kps = sample['src_kps'].numpy()
            trg_kps = sample['trg_kps'].numpy()
            trg_bbox = sample['trg_bbox']
            preds = []
            for i in range(src_kps.shape[0]):
                px, py = pixel_to_patch_coord(src_kps[i,0], src_kps[i,1], src_sz, patch_size, resized_size)
                sim = F.cosine_similarity(sf[0, py, px, :].unsqueeze(0), tf_flat, dim=1)
                mx, my = find_best_match_argmax(sim, W)
                rx, ry = patch_to_pixel_coord(mx, my, tgt_sz, patch_size, resized_size)
                preds.append([rx, ry])
            pcks = {}
            for thr in thresholds:
                pck, _, _ = compute_pck_spair71k(preds, trg_kps.tolist(), trg_bbox, thr)
                pcks[thr] = pck
            per_img.append({'category': sample['category'], 'pck_scores': pcks})
            if (idx + 1) % 200 == 0:
                print(f'  {idx+1} pairs...')
    return per_img

def save_exp_results(per_img, name, thresholds, results_dir):
    stats = {}
    for thr in thresholds:
        pcks = [m['pck_scores'][thr] for m in per_img]
        stats[f'pck@{thr:.2f}'] = {'mean': float(np.mean(pcks)), 'std': float(np.std(pcks))}
        print(f'  PCK@{thr:.2f}: {np.mean(pcks):.2f}% ± {np.std(pcks):.2f}%')
    out = {'name': name, 'n_pairs': len(per_img), 'stats': stats}
    path = os.path.join(results_dir, f'{name}.json')
    with open(path, 'w') as f: json.dump(out, f, indent=2)
    print(f'  Saved -> {path}')
    return stats

print('Utility functions loaded.')

In [ ]:
# Cell 4 — Load DINOv2 + datasets
from src.models.dinov2.dinov2.models.vision_transformer import vit_base as dinov2_vit_base

def load_dinov2():
    m = dinov2_vit_base(
        img_size=(518, 518), patch_size=14,
        num_register_tokens=0, block_chunks=0, init_values=1.0
    ).to(device)
    ckpt = torch.load(DINOV2_W, map_location=device)
    m.load_state_dict(ckpt, strict=True)
    return m

train_ds = SPairDataset(PAIR_ANN_PATH, LAYOUT_PATH, IMAGE_PATH, DATASET_SIZE, PCK_ALPHA, datatype='trn')
val_ds   = SPairDataset(PAIR_ANN_PATH, LAYOUT_PATH, IMAGE_PATH, DATASET_SIZE, PCK_ALPHA, datatype='val')
test_ds  = SPairDataset(PAIR_ANN_PATH, LAYOUT_PATH, IMAGE_PATH, DATASET_SIZE, PCK_ALPHA, datatype='test')
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=2)
print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
# Cell 5 — LoRA rank ablation (r = 2, 4, 8, 16)
rank_list = [2, 4, 8, 16]
lora_results = []

for r in rank_list:
    print(f'\n=== LoRA r={r} ===')
    model_lora = load_dinov2()
    inject_lora(model_lora, r=r, alpha=float(r), target_modules=('qkv', 'proj'))
    params = count_trainable_params(model_lora)
    print(f'  Trainable: {params["trainable"]:,} / {params["total"]:,} ({params["percentage"]}%)')

    exp_dir = os.path.join(WEIGHTS_DIR, f'r{r}')
    os.makedirs(exp_dir, exist_ok=True)

    best_pck, hist = train_with_scheduler(
        model_lora, train_loader, val_ds, device,
        img_size=IMG_SIZE, patch_size=PATCH_SIZE, temperature=TEMPERATURE,
        lr=LR, num_epochs=NUM_EPOCHS, warmup_steps=100, patience=PATIENCE,
        results_dir=exp_dir
    )

    # Evaluate on test set using best checkpoint
    ckpt = torch.load(os.path.join(exp_dir, 'best_checkpoint.pth'), map_location=device)
    model_lora.load_state_dict(ckpt['model_state_dict'])
    model_lora.eval()
    res = evaluate_model(model_lora, test_ds, device, THRESHOLDS, PATCH_SIZE, IMG_SIZE)
    stats = save_exp_results(res, f'lora_r{r}', THRESHOLDS, RESULTS_DIR)

    lora_results.append({
        'r': r,
        'trainable_params': params['trainable'],
        'pct': params['percentage'],
        'val_pck': best_pck,
        'stats': stats
    })
    del model_lora
    torch.cuda.empty_cache()

In [ ]:
# Cell 6 — Block-unfreezing baseline for comparison (4 blocks)
print('=== Block-unfreezing baseline (4 blocks) ===')
model_blocks = load_dinov2()
unfreeze_last_n_blocks(model_blocks, 4)
params_b = count_trainable_params(model_blocks)
print(f'Trainable: {params_b["trainable"]:,} ({params_b["percentage"]}%)')

block_dir = os.path.join(RESULTS_DIR, 'blocks4')
os.makedirs(block_dir, exist_ok=True)

best_pck_b, hist_b = train_with_scheduler(
    model_blocks, train_loader, val_ds, device,
    img_size=IMG_SIZE, patch_size=PATCH_SIZE, temperature=TEMPERATURE,
    lr=1e-5, num_epochs=NUM_EPOCHS, warmup_steps=100, patience=PATIENCE,
    results_dir=block_dir
)
ckpt_b = torch.load(os.path.join(block_dir, 'best_checkpoint.pth'), map_location=device)
model_blocks.load_state_dict(ckpt_b['model_state_dict'])
model_blocks.eval()
res_b = evaluate_model(model_blocks, test_ds, device, THRESHOLDS, PATCH_SIZE, IMG_SIZE)
stats_b = save_exp_results(res_b, 'blocks_4', THRESHOLDS, RESULTS_DIR)
del model_blocks
torch.cuda.empty_cache()

In [ ]:
# Cell 7 — Summary table
rows = []
for r_res in lora_results:
    r = r_res['r']
    s = r_res['stats']
    rows.append({
        'Method': f'LoRA r={r}',
        'Trainable (%)': f"{r_res['pct']:.4f}%",
        'PCK@0.05': f"{s.get('pck@0.05',{}).get('mean',0):.2f}%",
        'PCK@0.10': f"{s.get('pck@0.10',{}).get('mean',0):.2f}%",
        'PCK@0.20': f"{s.get('pck@0.20',{}).get('mean',0):.2f}%",
    })
rows.append({
    'Method': 'Block-unfreeze (4 blocks)',
    'Trainable (%)': f"{params_b['percentage']:.4f}%",
    'PCK@0.05': f"{stats_b.get('pck@0.05',{}).get('mean',0):.2f}%",
    'PCK@0.10': f"{stats_b.get('pck@0.10',{}).get('mean',0):.2f}%",
    'PCK@0.20': f"{stats_b.get('pck@0.20',{}).get('mean',0):.2f}%",
})
df = pd.DataFrame(rows)
print('\n=== Step 4a Results: LoRA vs Block-Unfreezing ===')
print(df.to_string(index=False))
df.to_csv(f'{RESULTS_DIR}/step4a_summary.csv', index=False)
print(f'Saved to {RESULTS_DIR}/step4a_summary.csv')

# Save best LoRA model
best_r = max(lora_results, key=lambda x: x['val_pck'])['r']
best_lora_ckpt = os.path.join(WEIGHTS_DIR, f'r{best_r}', 'best_checkpoint.pth')
best_lora_dst  = os.path.join(WEIGHTS_DIR, 'best_lora.pth')
import shutil
if os.path.exists(best_lora_ckpt):
    shutil.copy(best_lora_ckpt, best_lora_dst)
    print(f'Best LoRA (r={best_r}) saved -> {best_lora_dst}')